# PCA Example (Mall Customers Dataset)

**Goal: Understand the variance structure of the Mall Customers data and visualise K-Means clusters in principal component space.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/unsupervised/ at the repo root
SRC_UNSUP = os.path.join(REPO_ROOT, 'src', 'unsupervised')
sys.path.insert(0, SRC_UNSUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
from pca import PCA
from k_means_clustering import KMeans
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv(os.path.join(DATA_DIR, 'Mall_Customers.csv'))
MALL_FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X_raw = mall[MALL_FEATURES].values.astype(float)
X = StandardScaler().fit_transform(X_raw)
print(f"Dataset loaded: {mall.shape[0]} samples.")

## 2. Fit PCA

In [ ]:
pca = PCA(n_components=3).fit(X)
X_pca = pca.transform(X)
print(f'Explained variance ratio: {pca.explained_variance_ratio_.round(4)}')
print(f'Total explained:          {pca.explained_variance_ratio_.sum()*100:.1f}%')
print(f'Reconstruction error:     {pca.reconstruction_error(X):.8f}')

## 3. K-Means Cluster Labels for Colouring

In [ ]:
km = KMeans(k=5, init='k-means++', n_init=5, random_state=42).fit(X)
labels_km = km.labels_
cmap = plt.cm.get_cmap('tab10', 5)
print(f'Cluster sizes: {dict(zip(*np.unique(labels_km, return_counts=True)))}')

## 4. Results and Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(1,4), pca.explained_variance_ratio_, color=['steelblue','darkorange','seagreen'], edgecolor='white')
axes[0].plot(range(1,4), np.cumsum(pca.explained_variance_ratio_), 'o-', color='red', lw=1.5, label='Cumulative')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA - Scree Plot', fontweight='bold'); axes[0].legend()

for c in range(5):
    mask = labels_km==c
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1], color=cmap(c), s=30, alpha=0.75, label=f'Cluster {c}')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('PCA - PC1 vs PC2 (K-Means colours)', fontweight='bold'); axes[1].legend(fontsize=7)

loadings = pca.components_.T
for i, feat in enumerate(MALL_FEATURES):
    axes[2].arrow(0, 0, loadings[i,0], loadings[i,1], head_width=0.03, head_length=0.02, color=plt.cm.tab10(i), lw=2)
    axes[2].text(loadings[i,0]*1.12, loadings[i,1]*1.12, feat, fontsize=9, color=plt.cm.tab10(i), ha='center')
circle = plt.Circle((0,0),1,fill=False,color='gray',linestyle='--',lw=0.8)
axes[2].add_patch(circle)
axes[2].set_xlim(-1.3,1.3); axes[2].set_ylim(-1.3,1.3)
axes[2].set_xlabel('PC1 Loading'); axes[2].set_ylabel('PC2 Loading')
axes[2].set_title('PCA - Feature Biplot', fontweight='bold')
axes[2].axhline(0,color='gray',lw=0.5); axes[2].axvline(0,color='gray',lw=0.5)
plt.tight_layout(); plt.show()

## 5. Analysis

**Explained variance: PC1=44.3%, PC2=33.3%, PC3=22.4% — total=100%**

With only 3 features, all 3 principal components are needed to fully represent the data (reconstruction error = 0.0). The interesting finding is how the variance is distributed: PC1 captures the most information (44%) but the remaining variance is spread fairly evenly across PC2 and PC3. This indicates the three features (Age, Income, Spending Score) are not strongly correlated — they each contribute meaningfully to the data's structure.

**The biplot** reveals the relationships between features and principal components. Features pointing in the same direction are positively correlated; opposite directions indicate negative correlation. Typically, Income and Spending Score point in different directions along PC1 (confirming their weak correlation — wealthy people don't necessarily spend more), while Age often loads strongly onto PC2 (separating young from older customers).

**The PC1 vs PC2 scatter coloured by K-Means clusters** validates the clustering. If the clusters are well-separated in PCA space, the K-Means partition is capturing genuine structure rather than arbitrary divisions. Cluster 5 (young, wealthy, high spend) and Cluster 2 (older, low income, low spend) should appear in opposite corners of the plot — maximum spread confirms these are genuinely different customer types.

**Reconstruction error of exactly 0** confirms the implementation is mathematically correct — keeping all 3 components of 3-dimensional data is lossless by definition.

**Key takeaway:** PCA on this 3-feature dataset is primarily a visualisation tool rather than a compression tool. The scree plot confirms no single component dominates, meaning all three features carry independent information and none can be safely discarded without losing meaningful structure.